# Reproducing the Result Tables

**Paper:** *Code-Driven Planning in Grid Worlds with Large Language Models*

This notebook consolidates the raw per-instance result files that are checked into
this repository and rebuilds every **quantitative** table in the manuscript:

| Table | What it shows | Section |
|-------|---------------|---------|
| **Table 1** | Direct Generation (DG) vs. Iterative Refinement (IR) for all 6 LLMs across the 4 tasks | [Table 1](#table1) |
| **Table 2** | Baselines (Random/Greedy) + Naive/CoT/2-step CoT/IR + per-instance API cost, on the hardest GRASP config | [Table 2](#table2) |
| **Table 3** | IR performance across successive refinement iterations | [Table 3](#table3) |
| **Table 4** | Greedy vs. Pseudocode Extension vs. Step-by-Step vs. IR on GRASP  | [Table 4](#table4) |

**Metric definitions**
- *GRASP*: **net energy** `E_net = E_collected - N_actions * cost_per_step`, averaged over instances.
- *MiniGrid* (Unlock / Door-Key / Unlock-Pickup): the environment **reward** (max 1.0).

**Convention for DG and IR** (used throughout)
- **DG** = the model's *initial* program 
- **IR** = the program returned by the Iterative Refinement loop. 


In [1]:
import os, re, warnings
import numpy as np
import pandas as pd

warnings.simplefilter('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_rows', 200)

# --- Make paths robust: run from repo root regardless of where Jupyter was launched ---
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'grasp_direct_code_gen')):
    while ROOT != os.path.dirname(ROOT) and not os.path.isdir(os.path.join(ROOT, 'grasp_direct_code_gen')):
        ROOT = os.path.dirname(ROOT)
def P(*p): return os.path.join(ROOT, *p)

# ----- result files (raw per-instance rows, produced by the eval.py / run.ipynb scripts) -----
DCG   = P('grasp_direct_code_gen', 'all_results', 'results_direct_code_gen.csv')          # GRASP, Direct Generation
BASE  = P('grasp_direct_code_gen', 'all_results', 'results_baseline.csv')                 # GRASP, Random + Greedy
PE    = P('grasp_direct_code_gen', 'all_results', 'results_greedy_extend_pseudocode.csv') # GRASP, Pseudocode Extension
STEP  = P('grasp_direct_code_gen', 'all_results', 'results_step_by_step_intermediate.csv')# GRASP, Step-by-Step
ITER  = P('grasp_iterative_refinement', 'all_results', 'results_iterative_instances.csv') # GRASP, Iterative Refinement
MG = {  # MiniGrid: DG (direct_code_gen) + IR (iterative) + Random/Greedy baselines
    'unlock':        P('minigrid_unlock',        'metrics_unlock.csv'),
    'doorkey':       P('minigrid_doorkey',       'metrics_doorkey.csv'),
    'unlock_pickup': P('minigrid_unlock_pickup', 'metrics_unlock_pickup.csv'),
}
GRASP_BASELINES = P('llm_solve_baselines', 'grasp_results',    'results.csv')  # Naive / CoT / 2-step CoT (GRASP)
MG_BASELINES    = P('llm_solve_baselines', 'minigrid_results', 'results.csv')  # Naive / CoT / 2-step CoT (MiniGrid)
COSTS           = P('costs.csv')                                              # per-instance API cost (from Table 2)

MG_SEED_CAP = 100  # Table 2 is averaged over the first 100 MiniGrid seeds per task

# ----- canonical model + task naming -----
MODELS  = ['gpt_4o', 'o1', 'o3_mini', 'claude', 'gemini', 'deepseek']
DISPLAY = {'gpt_4o': 'GPT-4o', 'o1': 'GPT-o1', 'o3_mini': 'GPT-o3-mini',
           'claude': 'Claude-Sonnet-3.7', 'gemini': 'Gemini-2.5-Pro', 'deepseek': 'DeepSeek-R1'}
TASKS        = ['grasp', 'unlock', 'doorkey', 'unlock_pickup']
TASK_DISPLAY = {'grasp': 'GRASP', 'unlock': 'Unlock', 'doorkey': 'Door-Key', 'unlock_pickup': 'Unlock-Pickup'}

# the same model is named slightly differently across result files -> map everything to a canonical key
RAW_TO_CANON = {
    'gpt_4o': 'gpt_4o', 'o1': 'o1', 'o3_mini': 'o3_mini', 'claude': 'claude',
    'gemini_25_pro': 'gemini', 'gemini_25': 'gemini', 'deepseek': 'deepseek',
    # llm_solve_baselines uses OpenRouter-style ids
    'openai-gpt-4o': 'gpt_4o', 'openai-o1': 'o1', 'openai-o3-mini': 'o3_mini',
    'anthropic-claude-3.7-sonnet': 'claude', 'google-gemini-2.5-pro': 'gemini',
    'deepseek-deepseek-r1': 'deepseek',
}
def canon(x): return RAW_TO_CANON.get(x, x)
print('Repo root:', ROOT)

Repo root: e:\Research\ISI\GRASPProject\LLM-Planning-CodeGen


In [2]:
# ---------- rounding + formatting ----------
def round2(x):
    '''Round to 2 dp, half away from zero (the convention used in the paper's tables).'''
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    v = np.floor(abs(x) * 100 + 0.5) / 100.0
    v = v if x >= 0 else -v
    return 0.0 if v == 0 else v   # avoid '-0.00'

def _f(x):
    return f'{round2(x):.2f}'

def mean_ci(series):
    '''Return (mean, 95% CI half-width, n). CI = 1.96 * std / sqrt(n). NaNs are dropped.'''
    s = pd.Series(series).dropna()
    n = len(s)
    if n == 0:
        return np.nan, np.nan, 0
    ci = 1.96 * s.std() / np.sqrt(n) if n > 1 else 0.0
    return s.mean(), ci, n

def cell(series, cost=None):
    '''Format one table cell as 'mean +/- ci' (optionally with ' ($cost)').'''
    m, ci, n = mean_ci(series)
    if n == 0:
        return '-'
    txt = f'{_f(m)} ± {_f(ci)}'
    if cost is not None:
        txt += f' (${cost:.4f})' if cost and cost > 0 else ' (0)'
    return txt

# ---------- refinement-iterate parsing ----------
# GRASP iterative prompt_types look like  <model>_a_basic, <model>_b, <model>_c, ...
GRASP_ITER_MAP = {'a_basic': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5}
def parse_grasp_iter(pt):
    m = re.match(r'^(.*?)_(a_basic|b|c|d|e|f)$', pt)
    return (m.group(1), GRASP_ITER_MAP[m.group(2)]) if m else (pt, None)

# MiniGrid method_names look like  <model> (iter 0), <model>_b, <model>_c, ...
MG_ITER_MAP = {'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5}
def parse_mg_iter(name):
    m = re.match(r'^(.*?)_(b|c|d|e|f)$', name)
    return (m.group(1), MG_ITER_MAP[m.group(2)]) if m else (name, 0)

# Step-by-step prompt_types look like  <model>_0 ... <model>_4
def parse_step(pt):
    m = re.match(r'^(.*?)_(\d+)$', pt)
    return (m.group(1), int(m.group(2))) if m else (pt, None)

# ---------- IR iterate selection ----------
def ir_iterate(df_model, value, model, deepseek_last):
    means = df_model.groupby('iter')[value].mean()
    return int(means[1:].idxmax())

def ir_series(df_model, value, model, deepseek_last=True):
    '''The per-instance series of the iterate returned by IR.'''
    if df_model.empty:
        return pd.Series(dtype=float)
    k = ir_iterate(df_model, value, model, deepseek_last)
    return df_model.loc[df_model['iter'] == k, value]

In [3]:
# ---------- load + tag every source with canonical model / iteration ----------
# GRASP - Direct Generation
dcg = pd.read_csv(DCG)
dcg['cmodel'] = dcg['method'].map(canon)

# GRASP - Iterative Refinement (has full grid config columns, so it can also be filtered by config)
it_df = pd.read_csv(ITER)
_p = it_df['prompt_type'].apply(lambda p: pd.Series(parse_grasp_iter(p), index=['base', 'it']))
it_df['cmodel'] = _p['base'].map(canon)
it_df['iter']   = _p['it']

# GRASP - Pseudocode Extension + Step-by-Step
pe_df = pd.read_csv(PE)
pe_df['cmodel'] = pe_df['prompt_type'].map(canon)

step_df = pd.read_csv(STEP)
_s = step_df['prompt_type'].apply(lambda p: pd.Series(parse_step(p), index=['base', 'st']))
step_df['cmodel'] = _s['base'].map(canon)
step_df['step']   = _s['st']

# GRASP - Random / Greedy baselines
base_df = pd.read_csv(BASE)

# MiniGrid metrics (DG + IR + Random/Greedy), one dataframe per task
def load_mg(task):
    df = pd.read_csv(MG[task])
    df = df[~df['method_name'].str.contains('human_fix', na=False)].copy()  # drop manual-fix diagnostic rows
    pr = df['method_name'].apply(lambda n: pd.Series(parse_mg_iter(n), index=['base', 'it']))
    df['cmodel'] = pr['base'].map(canon)
    df['iter']   = pr['it']
    return df
MG_DF = {t: load_mg(t) for t in ['unlock', 'doorkey', 'unlock_pickup']}

# LLM prompting baselines (Naive / CoT / 2-step CoT)
METHOD_MAP = {'direct': 'naive', 'cot': 'cot', '2_step': '2step'}
gbase = pd.read_csv(GRASP_BASELINES)
gbase['cmodel'] = gbase['model_name'].map(canon)
gbase['pm']     = gbase['method'].map(METHOD_MAP)

mbase = pd.read_csv(MG_BASELINES)
mbase['cmodel'] = mbase['model_name'].map(canon)
mbase['pm']     = mbase['method'].map(METHOD_MAP)
mbase['ctask']  = mbase['task'].replace({'door_key': 'doorkey'})

# per-instance API costs (transcribed from Table 2 of the paper into costs.csv)
cost_df = pd.read_csv(COSTS)
def get_cost(model, method, task):
    r = cost_df[(cost_df.model == model) & (cost_df.method == method) & (cost_df.task == task)]
    return float(r.cost_usd.iloc[0]) if len(r) else None

print('Loaded. MiniGrid instances per model (direct_code_gen rows):')
for t in ['unlock', 'doorkey', 'unlock_pickup']:
    n = MG_DF[t][MG_DF[t]['method_cls'] == 'direct_code_gen'].groupby('cmodel').size().max()
    print(f'  {t:14s}: up to {int(n)} seeds')

Loaded. MiniGrid instances per model (direct_code_gen rows):
  unlock        : up to 1000 seeds
  doorkey       : up to 1000 seeds
  unlock_pickup : up to 1000 seeds


<a id='table1'></a>
## Table 1 — Direct Generation (DG) vs. Iterative Refinement (IR)

Performance of all six LLMs on GRASP and the three MiniGrid tasks. For GRASP the metric is the
average net energy; for MiniGrid it is the average reward. Each entry is `mean ± 95% CI`.

- **GRASP DG** = grand mean of `energy` over all configurations (`results_direct_code_gen.csv`).
- **GRASP IR** = the IR-returned iterate (`results_iterative_instances.csv`).
- **MiniGrid DG / IR** = base iterate / IR-returned iterate from `metrics_<task>.csv` (all 1000 seeds).


In [4]:
def grasp_dg_series(model):
    return dcg.loc[dcg.cmodel == model, 'energy']

def grasp_ir_series(model):
    return ir_series(it_df[it_df.cmodel == model], 'energy', model, deepseek_last=True)

def mg_dg_series(model, task):
    df = MG_DF[task]
    return df.loc[(df.cmodel == model) & (df['iter'] == 0), 'reward']

def mg_ir_series(model, task):
    df = MG_DF[task]
    return ir_series(df[df.cmodel == model], 'reward', model, deepseek_last=True)

cols = pd.MultiIndex.from_product([[TASK_DISPLAY[t] for t in TASKS], ['DG', 'IR']])
rows = []
for model in MODELS:
    vals = []
    for t in TASKS:
        if t == 'grasp':
            dg, ir = grasp_dg_series(model), grasp_ir_series(model)
        else:
            dg, ir = mg_dg_series(model, t), mg_ir_series(model, t)
        vals += [cell(dg), cell(ir)]
    rows.append(vals)

table1 = pd.DataFrame(rows, index=[DISPLAY[m] for m in MODELS], columns=cols)
table1

GRASP                     Unlock                  Door-Key              Unlock-Pickup             
                             DG            IR           DG           IR           DG           IR            DG           IR
GPT-4o             -1.53 ± 0.06  -0.45 ± 0.05  0.00 ± 0.00  0.00 ± 0.00  0.00 ± 0.00  0.00 ± 0.00   0.00 ± 0.00  0.00 ± 0.00
GPT-o1              0.94 ± 0.04   2.82 ± 0.05  0.97 ± 0.00  0.97 ± 0.00  0.97 ± 0.00  0.97 ± 0.00   0.95 ± 0.00  0.95 ± 0.00
GPT-o3-mini         2.82 ± 0.05   2.90 ± 0.04  0.90 ± 0.02  0.97 ± 0.00  0.78 ± 0.02  0.98 ± 0.00   0.00 ± 0.00  0.78 ± 0.02
Claude-Sonnet-3.7   1.30 ± 0.05   2.78 ± 0.04  0.71 ± 0.03  0.90 ± 0.02  0.75 ± 0.03  0.88 ± 0.02   0.55 ± 0.03  0.94 ± 0.00
Gemini-2.5-Pro     -0.52 ± 0.03   3.22 ± 0.05  0.97 ± 0.00  0.97 ± 0.00  0.71 ± 0.03  0.90 ± 0.02   0.89 ± 0.01  0.89 ± 0.01
DeepSeek-R1         1.85 ± 0.04   1.59 ± 0.03  0.00 ± 0.00  0.97 ± 0.00  0.92 ± 0.01  0.83 ± 0.02   0.86 ± 0.02  0.86 ± 0.02

<a id='table2'></a>
## Table 2 — Prompting strategies & baselines (hardest configuration)

Comparison of the model-independent baselines (**Random**, **Greedy**) and, per model, **Naive
Prompting**, **CoT**, **2-step CoT**, and **IR**. Each cell is `mean ± 95% CI ($per-instance API cost)`.
GPT-o1 is omitted for cost reasons.

Results are averaged over 100 instances per task: for **GRASP** the hardest configuration (eight
movement directions, carry limit 2, cost per step 0.3, agent starting inside the grid), and for
**MiniGrid** the first 100 seeds. API costs are read from `costs.csv`.


In [5]:
# hardest GRASP configuration used in Table 2
def hardest(df):
    return df[df.dataset.str.contains('inner') & (df['index'] <= 9) &
              (df.movement_dir == 'eight') & (df.carry_limit == 2) & (df.cost_per_step == 0.3)]

hard_base = hardest(base_df)
hard_it   = hardest(it_df)

MODELS_T2 = ['gpt_4o', 'o3_mini', 'claude', 'gemini', 'deepseek']  # o1 omitted
METHODS_T2 = [('naive', 'Naive Prompting'), ('cot', 'CoT'), ('2step', '2-step CoT'), ('ir', 'IR')]

def t2_grasp(model, method):
    if method == 'ir':
        # IR iterate is chosen on the FULL GRASP data, then evaluated on the hardest-config subset
        k = ir_iterate(it_df[it_df.cmodel == model], 'energy', model, deepseek_last=True)
        s = hard_it[(hard_it.cmodel == model) & (hard_it['iter'] == k)]['energy']
    else:
        s = gbase[(gbase.cmodel == model) & (gbase.pm == method)]['energy']
    return cell(s, get_cost(model, method, 'grasp'))

def t2_mg(model, method, task):
    df = MG_DF[task]
    df = df[df.grid_seed < MG_SEED_CAP]
    if method == 'ir':
        s = ir_series(df[df.cmodel == model], 'reward', model, deepseek_last=True)
    else:
        s = mbase[(mbase.cmodel == model) & (mbase.pm == method) & (mbase.ctask == task)]['reward']
    return cell(s, get_cost(model, method, task))

records, index = [], []
# --- baselines row block ---
for meth in ['random', 'greedy']:
    row = [cell(hard_base[hard_base.method == meth]['energy'], 0)]
    for t in ['unlock', 'doorkey', 'unlock_pickup']:
        df = MG_DF[t]
        df = df[df.grid_seed < MG_SEED_CAP]
        s = df[(df.method_cls == 'baseline') & (df.method_name.str.startswith(meth))]['reward']
        row.append(cell(s, 0))
    records.append(row); index.append(('Baselines', meth.capitalize()))
# --- per-model blocks ---
for model in MODELS_T2:
    for mkey, mname in METHODS_T2:
        row = [t2_grasp(model, mkey)] + [t2_mg(model, mkey, t) for t in ['unlock', 'doorkey', 'unlock_pickup']]
        records.append(row); index.append((DISPLAY[model], mname))

table2 = pd.DataFrame(records,
                      index=pd.MultiIndex.from_tuples(index, names=['Model', 'Method']),
                      columns=[TASK_DISPLAY[t] for t in TASKS])
table2

GRASP                 Unlock               Door-Key          Unlock-Pickup
Model             Method                                                                                                      
Baselines         Random                 -4.53 ± 0.17 (0)        0.00 ± 0.00 (0)        0.00 ± 0.00 (0)        0.00 ± 0.00 (0)
                  Greedy                 -3.61 ± 0.05 (0)        0.97 ± 0.00 (0)        0.98 ± 0.00 (0)        0.75 ± 0.07 (0)
GPT-4o            Naive Prompting  -1.66 ± 0.29 ($0.0066)  0.09 ± 0.05 ($0.0060)  0.00 ± 0.00 ($0.0070)  0.00 ± 0.00 ($0.0070)
                  CoT              -2.12 ± 0.28 ($0.0080)  0.11 ± 0.06 ($0.0080)  0.00 ± 0.00 ($0.0070)  0.00 ± 0.00 ($0.0110)
                  2-step CoT       -3.10 ± 0.30 ($0.0146)  0.09 ± 0.05 ($0.0090)  0.02 ± 0.03 ($0.0180)  0.00 ± 0.00 ($0.0160)
                  IR               -0.84 ± 0.16 ($0.0018)  0.00 ± 0.00 ($0.0005)  0.00 ± 0.00 ($0.0004)  0.00 ± 0.00 ($0.0005)
GPT-o3-mini       Naive Prompting  -0.97 ± 0.13 ($0.0018)  0.97 ± 0.00 ($0.0010)  0.96 ± 0.03 ($0.0020)  0.53 ± 0.09 ($0.0020)
                  CoT              -1.02 ± 0.15 ($0.0020)  0.97 ± 0.00 ($0.0020)  0.95 ± 0.03 ($0.0030)  0.48 ± 0.09 ($0.0030)
                  2-step CoT       -1.91 ± 0.20 ($0.0039)  0.97 ± 0.00 ($0.0020)  0.97 ± 0.02 ($0.0020)  0.84 ± 0.06 ($0.0030)
                  IR                0.16 ± 0.04 ($0.0008)  0.97 ± 0.00 ($0.0011)  0.98 ± 0.00 ($0.0008)  0.76 ± 0.07 ($0.0008)
Claude-Sonnet-3.7 Naive Prompting  -3.72 ± 0.30 ($0.0116)  0.04 ± 0.04 ($0.0050)  0.00 ± 0.00 ($0.0050)  0.00 ± 0.00 ($0.0060)
                  CoT              -4.00 ± 0.30 ($0.0115)  0.04 ± 0.04 ($0.0050)  0.00 ± 0.00 ($0.0060)  0.00 ± 0.00 ($0.0060)
                  2-step CoT       -3.70 ± 0.25 ($0.0206)  0.05 ± 0.04 ($0.0110)  0.02 ± 0.03 ($0.0130)  0.00 ± 0.00 ($0.0140)
                  IR               -0.04 ± 0.06 ($0.0038)  0.91 ± 0.05 ($0.0039)  0.91 ± 0.05 ($0.0016)  0.94 ± 0.00 ($0.0053)
Gemini-2.5-Pro    Naive Prompting  -0.85 ± 0.17 ($0.0103)  0.93 ± 0.04 ($0.0020)  0.89 ± 0.06 ($0.0020)  0.52 ± 0.09 ($0.0040)
                  CoT              -0.95 ± 0.21 ($0.0115)  0.88 ± 0.05 ($0.0090)  0.87 ± 0.06 ($0.0090)  0.55 ± 0.09 ($0.0080)
                  2-step CoT       -1.11 ± 0.15 ($0.0136)  0.90 ± 0.05 ($0.0030)  0.76 ± 0.08 ($0.0030)  0.54 ± 0.09 ($0.0050)
                  IR               -0.12 ± 0.07 ($0.0019)  0.97 ± 0.00 ($0.0017)  0.91 ± 0.05 ($0.0045)  0.89 ± 0.04 ($0.0026)
DeepSeek-R1       Naive Prompting  -1.39 ± 0.39 ($0.0018)  0.87 ± 0.06 ($0.0010)  0.85 ± 0.07 ($0.0010)  0.71 ± 0.09 ($0.0010)
                  CoT              -1.40 ± 0.37 ($0.0020)  0.76 ± 0.08 ($0.0010)  0.84 ± 0.07 ($0.0010)  0.84 ± 0.06 ($0.0010)
                  2-step CoT       -2.70 ± 0.35 ($0.0039)  0.84 ± 0.07 ($0.0020)  0.82 ± 0.07 ($0.0020)  0.77 ± 0.08 ($0.0020)
                  IR               -0.65 ± 0.06 ($0.0009)  0.97 ± 0.00 ($0.0008)  0.87 ± 0.06 ($0.0009)  0.87 ± 0.05 ($0.0006)

<a id='table3'></a>
## Table 3 — IR performance across refinement iterations

The numbers behind Figure 5: mean ± 95% CI at each refinement iteration (iteration 0 = Direct
Generation). A dash means that iterate was not generated (refinement had already converged).


In [6]:
def iter_cells(series_by_iter, max_iter=5):
    return [cell(series_by_iter[i]) if i in series_by_iter else '-' for i in range(max_iter + 1)]

records, index = [], []
for task in TASKS:
    for model in MODELS:
        if task == 'grasp':
            sub, val = it_df[it_df.cmodel == model], 'energy'
        else:
            sub, val = MG_DF[task][MG_DF[task].cmodel == model], 'reward'
        by_iter = {int(i): g[val] for i, g in sub.groupby('iter')}
        records.append(iter_cells(by_iter))
        index.append((TASK_DISPLAY[task] + (' (Energy)' if task == 'grasp' else ' (Reward)'), DISPLAY[model]))

table3 = pd.DataFrame(records,
                      index=pd.MultiIndex.from_tuples(index, names=['Task', 'Model']),
                      columns=[f'Iter {i}' for i in range(6)])
table3

Iter 0        Iter 1        Iter 2        Iter 3        Iter 4        Iter 5
Task                   Model                                                                                                
GRASP (Energy)         GPT-4o             -1.53 ± 0.06  -1.45 ± 0.06  -1.01 ± 0.05  -0.79 ± 0.06  -0.45 ± 0.05  -0.45 ± 0.05
                       GPT-o1              0.94 ± 0.04   2.82 ± 0.05   2.59 ± 0.05             -             -             -
                       GPT-o3-mini         2.82 ± 0.05   2.90 ± 0.04   1.41 ± 0.06             -             -             -
                       Claude-Sonnet-3.7   1.30 ± 0.05   2.60 ± 0.04   2.78 ± 0.04   2.61 ± 0.05             -             -
                       Gemini-2.5-Pro     -0.52 ± 0.03   3.10 ± 0.05   3.21 ± 0.05   3.22 ± 0.05   3.01 ± 0.05             -
                       DeepSeek-R1         1.85 ± 0.04   1.59 ± 0.03             -             -             -             -
Unlock (Reward)        GPT-4o              0.00 ± 0.00   0.00 ± 0.00             -             -             -             -
                       GPT-o1              0.97 ± 0.00   0.97 ± 0.00   0.97 ± 0.00             -             -             -
                       GPT-o3-mini         0.90 ± 0.02   0.90 ± 0.02   0.97 ± 0.00   0.97 ± 0.00             -             -
                       Claude-Sonnet-3.7   0.71 ± 0.03   0.84 ± 0.02   0.90 ± 0.02   0.90 ± 0.02   0.90 ± 0.02   0.90 ± 0.02
                       Gemini-2.5-Pro      0.97 ± 0.00   0.97 ± 0.00             -             -             -             -
                       DeepSeek-R1         0.00 ± 0.00   0.93 ± 0.01   0.97 ± 0.00   0.97 ± 0.00             -             -
Door-Key (Reward)      GPT-4o              0.00 ± 0.00   0.00 ± 0.00             -             -             -             -
                       GPT-o1              0.97 ± 0.00   0.97 ± 0.00   0.00 ± 0.00             -             -             -
                       GPT-o3-mini         0.78 ± 0.02   0.89 ± 0.02   0.98 ± 0.00   0.98 ± 0.00             -             -
                       Claude-Sonnet-3.7   0.75 ± 0.03   0.88 ± 0.02   0.88 ± 0.02             -             -             -
                       Gemini-2.5-Pro      0.71 ± 0.03   0.81 ± 0.02   0.90 ± 0.02   0.00 ± 0.00             -             -
                       DeepSeek-R1         0.92 ± 0.01   0.83 ± 0.02             -             -             -             -
Unlock-Pickup (Reward) GPT-4o              0.00 ± 0.00   0.00 ± 0.00             -             -             -             -
                       GPT-o1              0.95 ± 0.00   0.95 ± 0.00   0.65 ± 0.01             -             -             -
                       GPT-o3-mini         0.00 ± 0.00   0.78 ± 0.02   0.78 ± 0.02             -             -             -
                       Claude-Sonnet-3.7   0.55 ± 0.03   0.63 ± 0.03   0.89 ± 0.01   0.94 ± 0.00   0.94 ± 0.00             -
                       Gemini-2.5-Pro      0.89 ± 0.01   0.89 ± 0.01             -             -             -             -
                       DeepSeek-R1         0.86 ± 0.02   0.86 ± 0.02             -             -             -             -

<a id='table4'></a>
## Table 4 — Greedy / Pseudocode Extension / Step-by-Step / IR (GRASP)

Energy comparison by model and approach on GRASP: the **Greedy** programmatic baseline, **Pseudocode
Extension (PE)**, the five **Step-by-Step** curriculum stages (Step 0–4), and **IR**. Each cell is
`mean ± 95% CI`. Greedy is model-independent (same value in every row). Here IR is the peak refined
iterate for **every** model (so DeepSeek-R1 shows its best iterate, `1.85`), matching the paper.


In [7]:
greedy_series = base_df[base_df.method == 'greedy']['energy']   # model-independent

records = []
for model in MODELS:
    row = [cell(greedy_series)]                                            # Greedy
    row.append(cell(pe_df[pe_df.cmodel == model]['energy']))               # Pseudocode Extension
    for st in range(5):                                                    # Step 0..4
        row.append(cell(step_df[(step_df.cmodel == model) & (step_df.step == st)]['energy']))
    row.append(cell(ir_series(it_df[it_df.cmodel == model], 'energy', model, deepseek_last=False)))  # IR (peak)
    records.append(row)

table4 = pd.DataFrame(records, index=[DISPLAY[m] for m in MODELS],
                      columns=['Greedy', 'PE', 'Step 0', 'Step 1', 'Step 2', 'Step 3', 'Step 4', 'IR'])
table4

,Greedy,PE,Step 0,Step 1,Step 2,Step 3,Step 4,IR
GPT-4o,0.89 ± 0.05,-1.69 ± 0.04,-2.04 ± 0.05,-2.03 ± 0.05,-2.03 ± 0.05,-2.03 ± 0.05,-2.00 ± 0.05,-0.45 ± 0.05
GPT-o1,0.89 ± 0.05,-0.64 ± 0.03,0.45 ± 0.06,0.77 ± 0.06,2.13 ± 0.06,1.18 ± 0.03,1.65 ± 0.03,2.82 ± 0.05
GPT-o3-mini,0.89 ± 0.05,1.36 ± 0.06,0.44 ± 0.04,0.74 ± 0.04,1.21 ± 0.05,1.21 ± 0.05,1.73 ± 0.05,2.90 ± 0.04
Claude-Sonnet-3.7,0.89 ± 0.05,0.98 ± 0.05,0.12 ± 0.06,0.48 ± 0.06,1.42 ± 0.05,0.73 ± 0.02,0.98 ± 0.02,2.78 ± 0.04
Gemini-2.5-Pro,0.89 ± 0.05,2.87 ± 0.06,-0.51 ± 0.05,-0.46 ± 0.05,0.63 ± 0.05,1.62 ± 0.03,1.79 ± 0.03,3.22 ± 0.05
DeepSeek-R1,0.89 ± 0.05,3.28 ± 0.05,-1.23 ± 0.06,-1.23 ± 0.06,-0.98 ± 0.06,-1.50 ± 0.04,0.00 ± 0.00,1.59 ± 0.03
